In [ ]:
# 9_generate_portraits.ipynb
#
# Generates a DALL-E portrait image for each cluster/tribe row in the
# cluster CSV. Images are saved to api/data/portraits/ and referenced
# by the API via a portrait_url field.
#
# DEMO: set GROUP_FILTER to a single employment group (e.g. "Employed") to
# limit cost; set to None to generate for every group.
#
# Only generates images for rows that don't already have a portrait,
# so re-running is cheap after the first pass.
#
# Requires OPENAI_API_KEY in environment or .env file.

import sys, os, time, re, json
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path
from urllib.request import urlopen
from tqdm import tqdm

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES
from openai import OpenAI

# ── Load .env if present ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env', override=False)
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Export it or add to .env:\n"
        "  export OPENAI_API_KEY=sk-..."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Config ────────────────────────────────────────────────────────────────────
CLUSTER_CSV   = Path(f"../{DATA_FOLDER}/6_cluster/LA_london_clusters.csv")
PORTRAIT_DIR  = Path("../api/data/portraits")
PORTRAIT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_MODEL   = "dall-e-3"
IMAGE_SIZE    = "1024x1024"
MAX_RETRIES   = 3
RETRY_DELAY   = 5

# DEMO: only "Employed" clusters — set to None for all groups
GROUP_FILTER = "Employed"

# Same geography scope as 6_cluster (config_paths)
UNIT_FILTER = list(FOUR_LA_CODES) if USE_FOUR_LA_SUBSET else "london"

df = pd.read_csv(CLUSTER_CSV)
if UNIT_FILTER is None:
    pass
elif UNIT_FILTER == "london":
    df = df[df["unit_id"].astype(str).str.startswith("E09")]
elif isinstance(UNIT_FILTER, list):
    df = df[df["unit_id"].isin(set(UNIT_FILTER))]
else:
    raise ValueError("UNIT_FILTER must be None, 'london', or a list of LA codes")
print(f"Loaded {len(df)} cluster rows from {CLUSTER_CSV.name} [USE_FOUR_LA_SUBSET={USE_FOUR_LA_SUBSET}]")
if GROUP_FILTER is not None:
    df = df[df["group"] == GROUP_FILTER].copy()
    print(f"Filtered to group={GROUP_FILTER!r}: {len(df)} rows (demo mode)")


# ── Ethnicity label mapping (post-recode codes) ──────────────────────────────
ETHNICITY_DESCRIPTIONS = {
    "Not stated":              None,
    "White":                   "White British",
    "Mixed":                   "mixed heritage",
    "Indian":                  "Indian",
    "Pakistani / Bangladeshi": "South Asian (Pakistani or Bangladeshi)",
    "Other Asian":             "East or Southeast Asian",
    "Arab":                    "Middle Eastern or Arab",
    "Caribbean":               "Black Caribbean",
    "African":                 "Black African",
    "Other":                   None,
}


def portrait_filename(unit_id: str, tribe_label: str) -> str:
    """Deterministic filename from unit_id + tribe_label."""
    safe = re.sub(r'[^a-zA-Z0-9]+', '_', f"{unit_id}_{tribe_label}").strip('_').lower()
    return f"{safe}.png"


def portrait_path_for_row(row: pd.Series) -> Path:
    return PORTRAIT_DIR / portrait_filename(row["unit_id"], row["tribe_label"])


def portrait_missing(fpath: Path) -> bool:
    """True if we should generate: no file, or empty/corrupt zero-byte file."""
    if not fpath.is_file():
        return True
    try:
        return fpath.stat().st_size == 0
    except OSError:
        return True


def build_portrait_prompt(row: pd.Series) -> str:
    """Build a DALL-E prompt from the cluster profile."""
    age = row.get("Age in years")
    gender = row.get("Gender", "person")
    ethnicity_raw = row.get("Ethnic group", "")
    ethnicity = ETHNICITY_DESCRIPTIONS.get(ethnicity_raw)

    age_str = f"approximately {int(age)} years old" if pd.notna(age) else ""
    gender_str = gender if isinstance(gender, str) and gender in ("Male", "Female") else "person"
    eth_str = f", {ethnicity} ethnicity" if ethnicity else ""

    group = row.get("group", "")
    if group == "Retired":
        activity = "relaxed, casual clothing"
    elif group == "Student":
        activity = "casual student attire"
    elif group == "Unemployed":
        activity = "casual everyday clothing"
    elif group in ("Employed", "Self-employed"):
        activity = "smart-casual work attire"
    else:
        activity = "casual clothing"

    prompt = (
        f"A realistic photographic portrait of a British {gender_str.lower()}, "
        f"{age_str}{eth_str}, wearing {activity}. "
        f"Head and shoulders shot, natural lighting, neutral plain background, "
        f"looking directly at the camera with a natural expression. "
        f"High quality, photorealistic style."
    )
    return prompt


def generate_portrait(prompt: str, filepath: Path) -> bool:
    """Call DALL-E and save the image. Returns True on success."""
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.images.generate(
                model=IMAGE_MODEL,
                prompt=prompt,
                n=1,
                size=IMAGE_SIZE,
                quality="standard",
            )
            image_url = resp.data[0].url
            img_data = urlopen(image_url, timeout=60).read()
            filepath.write_bytes(img_data)
            return True
        except Exception as e:
            wait = RETRY_DELAY * (2 ** attempt)
            print(f"  Error: {e} — retrying in {wait}s")
            time.sleep(wait)
    return False


# ── Generate portraits (only rows with no file on disk yet) ───────────────────
df_pending = df[df.apply(lambda r: portrait_missing(portrait_path_for_row(r)), axis=1)]
already = len(df) - len(df_pending)
print(f"Already on disk (skipped): {already}  |  To generate: {len(df_pending)}")

generated = 0
failed = 0

for _, row in tqdm(df_pending.iterrows(), total=len(df_pending), desc="Generating portraits"):
    fpath = portrait_path_for_row(row)
    fname = fpath.name

    prompt = build_portrait_prompt(row)
    if generate_portrait(prompt, fpath):
        generated += 1
    else:
        failed += 1
        print(f"  FAILED: {fname}")

print(f"\nDone. Newly generated: {generated}, Skipped (already existed): {already}, Failed: {failed}")
print(f"Portraits directory: {PORTRAIT_DIR}")
print(f"Total files: {len(list(PORTRAIT_DIR.glob('*.png')))}")


Loaded 355 cluster rows from LA_london_clusters.csv
Filtered to group='Employed': 118 rows (demo mode)
Already on disk (skipped): 97  |  To generate: 21


Generating portraits:  67%|██████▋   | 14/21 [03:32<01:47, 15.41s/it]